In [ ]:
import os
# 현재 경로 출력
print(os.getcwd())

API_KEY='' # google api key

In [ ]:
from google.colab import output # colab output module 설정
output.enable_custom_widget_manager()

In [ ]:
!pip install google-maps-routes
!pip install polyline

In [ ]:
# google maps를 이용하여 쓰레기, 쓰레기통, 현재, 목표위치 시각화하는 코드. Colab에서 실행하는 것이 환경 세팅에 편하다.
import collections
collections.Iterable = collections.abc.Iterable
import gmaps
import json
from IPython.display import display

# 1. API 키 설정

gmaps.configure(api_key=API_KEY)

# 2. JSON 데이터 파일 로드
try:
    with open('temp_trash_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print("오류: temp_trash_data.json 파일을 찾을 수 없습니다.")
    data = {"trash": [], "bins": [], "current": None, "destination": None}

# 3. 위치 데이터 파싱
trash_locations = [(loc['lat'], loc['lng']) for loc in data.get('trash', [])]
bin_locations = [(loc['lat'], loc['lng']) for loc in data.get('bins', [])]

# ==================== 코드 변경점 1: current, destination 위치 파싱 ====================
current_location = tuple(data['current'].values()) if data.get('current') else None
destination_location = tuple(data['destination'].values()) if data.get('destination') else None
# =================================================================================

# 4. 지도 생성
map_center = (37.626, 127.078)
fig = gmaps.figure(center=map_center, zoom_level=15)

# 5. 쓰레기 위치 레이어 생성 (빨간색 점)
trash_layer = gmaps.symbol_layer(
    trash_locations,
    fill_color='rgba(255, 0, 0, 0.8)',
    stroke_color='rgba(255, 0, 0, 1)',
    scale=4,
    info_box_content=['쓰레기 위치' for _ in trash_locations]
)

# 6. 쓰레기통 위치 레이어 생성 (파란색 점)
bin_layer = gmaps.symbol_layer(
    bin_locations,
    fill_color='rgba(0, 0, 255, 0.8)',
    stroke_color='rgba(0, 0, 255, 1)',
    scale=6,
    info_box_content=['쓰레기통 위치' for _ in bin_locations]
)

# 7. 지도에 기본 레이어 추가
fig.add_layer(trash_layer)
fig.add_layer(bin_layer)

# ==================== 코드 변경점 2: current, destination 레이어 추가 ===================
# 현재 위치 레이어 (녹색 점)
if current_location:
    current_layer = gmaps.symbol_layer(
        [current_location],  # symbol_layer는 리스트 형태의 입력을 받음
        fill_color='rgba(0, 200, 0, 0.8)',  # 녹색
        stroke_color='rgba(0, 200, 0, 1)',
        scale=8,  # 다른 점들보다 크게 표시
        info_box_content='현재 위치'
    )
    fig.add_layer(current_layer)

# 목적지 위치 레이어 (검은색 점)
if destination_location:
    destination_layer = gmaps.symbol_layer(
        [destination_location], # symbol_layer는 리스트 형태의 입력을 받음
        fill_color='rgba(0, 0, 0, 0.8)',  # 검은색
        stroke_color='rgba(0, 0, 0, 1)',
        scale=8,  # 다른 점들보다 크게 표시
        info_box_content='목적지'
    )
    fig.add_layer(destination_layer)
# =================================================================================

# 8. 지도 표시
print("temp_trash_data.json 파일의 모든 위치 데이터를 지도에 시각화합니다.")
display(fig)

In [ ]:

# json data로부터 시작, 도착, 경유지 정보를 읽어 Routes API 요청을 생성하고 생성된 경로 결과를 시각화

"""
Routes API 동적 요청 테스트
- route_example.json 파일을 읽어 요청 바디를 생성
- 응답의 encoded polyline을 디코딩하여 (선택) gmaps로 시각화
필요: requests, polyline, (선택) gmaps, route_example.json 파일
"""

import os
import json  # <--- JSON 파일을 다루기 위해 추가
import requests
import polyline

# (선택) gmaps로 지도 시각화
USE_GMAPS = True
try:
    import collections, collections.abc  # py3.10 호환
    if not hasattr(collections, "Iterable"):
        collections.Iterable = collections.abc.Iterable
    import gmaps
except Exception:
    USE_GMAPS = False
    gmaps = None

# ===== 1) 설정 =====
# API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY") or "YOUR_GOOGLE_API_KEY"  # ← 실제 키로 바꾸거나 env 설정
ROUTES_URL = "https://routes.googleapis.com/directions/v2:computeRoutes"
ROUTE_DATA_FILE = "temp_data.json" # <--- 읽어올 파일 이름

# 필요한 필드만 받도록 필드마스크 지정
FIELD_MASK = (
    "routes.distanceMeters,"
    "routes.duration,"
    "routes.polyline.encodedPolyline"
)

def create_request_body_from_file(filepath):
    """JSON 파일을 읽어 Routes API 요청 본문 형식으로 변환합니다."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        raise RuntimeError(f"'{filepath}' 파일을 찾을 수 없습니다. 스크립트와 같은 위치에 파일을 생성하세요.")
    except json.JSONDecodeError:
        raise RuntimeError(f"'{filepath}' 파일의 JSON 형식이 올바르지 않습니다.")

    # JSON 데이터를 Routes API 형식에 맞게 변환
    request_body = {
        "origin": {
            "location": { "latLng": { "latitude": data["current"]["lat"], "longitude": data["current"]["lng"] } }
        },
        "destination": {
            "location": { "latLng": { "latitude": data["destination"]["lat"], "longitude": data["destination"]["lng"] } }
        },
        "intermediates": [
            { "location": { "latLng": { "latitude": stop["lat"], "longitude": stop["lng"] } } }
            for stop in data.get("stops", [])
        ],
        "travelMode": "WALK",
    }
    return request_body


def main():
    if not API_KEY or API_KEY == "YOUR_GOOGLE_API_KEY":
        raise RuntimeError("API 키를 설정하세요. 환경변수 GOOGLE_MAPS_API_KEY 또는 코드 상단 API_KEY 사용.")

    # --- 변경된 부분: JSON 파일로부터 요청 바디 생성 ---
    request_body = create_request_body_from_file(ROUTE_DATA_FILE)
    print(f"'{ROUTE_DATA_FILE}' 파일로부터 경로 정보를 읽었습니다.")
    # ----------------------------------------------

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK
    }

    # 2) Routes API 호출
    r = requests.post(ROUTES_URL, headers=headers, json=request_body, timeout=20)
    if r.status_code != 200:
        print("[Routes API] HTTP", r.status_code, r.text[:500])
        r.raise_for_status()

    data = r.json()
    routes = data.get("routes", [])
    if not routes:
        print("[Routes API] no routes. payload preview:", str(data)[:500])
        raise RuntimeError("Routes API가 경로를 반환하지 않았습니다.")

    route0 = routes[0]
    enc = route0.get("polyline", {}).get("encodedPolyline")
    dist_m = route0.get("distanceMeters")
    dur_s = route0.get("duration")

    print(f"[ROUTE] distance={dist_m} m, duration={dur_s}")
    if not enc:
        print("! 폴리라인이 없어 시각화는 생략합니다.")
        return

    # 3) 폴리라인 디코딩
    decoded = polyline.decode(enc)  # [(lat, lng), ...]
    print(f"[POLYLINE] points={len(decoded)} (첫 3개) {decoded[:3]}")

    # 4) (선택) gmaps로 시각화
    if USE_GMAPS:
        gmaps.configure(api_key=API_KEY)

        lines = [
            gmaps.Line(
                start=tuple(decoded[i]),
                end=tuple(decoded[i+1]),
                stroke_color="green",
                stroke_weight=5,
                stroke_opacity=0.85,
            )
            for i in range(len(decoded) - 1)
        ]
        layer = gmaps.drawing_layer(features=lines)

        # 지도에 표시할 마커 좌표 추출 (출발지, 경유지, 도착지)
        origin = (request_body["origin"]["location"]["latLng"]["latitude"],
                  request_body["origin"]["location"]["latLng"]["longitude"])
        destination = (request_body["destination"]["location"]["latLng"]["latitude"],
                       request_body["destination"]["location"]["latLng"]["longitude"])
        
        waypoints = [
            (waypoint["location"]["latLng"]["latitude"], waypoint["location"]["latLng"]["longitude"])
            for waypoint in request_body.get("intermediates", [])
        ]
        
        all_points = [origin] + waypoints + [destination]
        labels = ["S"] + [f"W{i+1}" for i in range(len(waypoints))] + ["D"]

        fig = gmaps.figure(center=origin, zoom_level=15)
        fig.add_layer(gmaps.marker_layer(all_points, label=labels))
        fig.add_layer(layer)

        print("\n지도 시각화(gmaps)")
        from IPython.display import display
        display(fig)
    else:
        print("(gmaps 미사용) 디코딩 좌표만 확인했습니다.")


if __name__ == "__main__":
    main()